# Verify an Experanto export (PSTH + integrity vs. datastore)

Sanity check for a MOZAIK → Experanto **export**, in two parts:

- **§1–6 (export only, plain numpy):** compute per-neuron PSTHs from the exported
  `responses/{spikes.npy, meta.yml}` + `screen/{combined_meta.json, timestamps.npy}` and check
  **stimulus-locking** (evoked vs. pre-stim baseline) — validates the export's spike↔screen timeline.
- **§7 (integrity vs. datastore):** compare export PSTHs against PSTHs computed directly from the
  source MOZAIK **datastore** (same neurons/binning). Needs one in-container step to produce
  `datastore_psths.npz` (see §7); the comparison itself is plain numpy.

> Exact `exported spikes == datastore spikes` equality is separately guaranteed by
> `mozaik tests/tools/test_experanto_export.py`. §7 is the response-level, timeline-aware cross-check.

**To use:** set `EXPERANTO_ROOT` / `N_TRIALS` / `OUTPUT_DIR` in the config cell, then Run All
(for §7, produce `datastore_psths.npz` first — see that section).

In [ ]:
# ============================ CONFIG — EDIT ME ============================
from pathlib import Path

# Directory containing trial0/, trial1/, ... each with responses/ and screen/.
EXPERANTO_ROOT = Path(
    "/mnt/lustre-grete/tmp/u18196/claude-897072/"
    "-mnt-vast-nhr-projects-nix00014-goirik-MOZAIK-new/"
    "2bb66c81-c220-485b-ba0d-0e8d20f118fe/scratchpad/SR_TEST3/export"
)
N_TRIALS = 3            # number of trial{t}/ dirs to load

BIN_SIZE_MS = 10        # PSTH bin width (ms)
PRE_STIM_MS = 100       # window before stimulus onset
POST_STIM_MS = 100      # window after stimulus offset
N_NEURONS = 20          # top-N most active neurons (by spike count in trial0)

# Must match the sim's RandomizedExperanto timing.
FRAME_DURATION_MS = 7.0
MOVIE_FRAME_DURATION_MS = 35.0

# Working dir: where §3 saves top_neuron_indices.npy (input to the datastore-PSTH step)
# and where §7 reads the datastore PSTHs. See §7 for how to produce datastore_psths.npz.
OUTPUT_DIR = Path(
    "/mnt/lustre-grete/tmp/u18196/claude-897072/"
    "-mnt-vast-nhr-projects-nix00014-goirik-MOZAIK-new/"
    "2bb66c81-c220-485b-ba0d-0e8d20f118fe/scratchpad/SR_TEST3/psth_out"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASTORE_PSTHS = OUTPUT_DIR / "datastore_psths.npz"   # produced by compute_psth_datastore.py (§7)
# ========================================================================

In [ ]:
import json
import numpy as np
import yaml
import matplotlib.pyplot as plt

assert (EXPERANTO_ROOT / "trial0").is_dir(), f"No trial0/ under {EXPERANTO_ROOT}"
print("Export:", EXPERANTO_ROOT)

## 1. Build the stimulus table from the screen metadata

In [ ]:
trial0 = EXPERANTO_ROOT / "trial0"
with open(trial0 / "screen" / "combined_meta.json") as f:
    combined_meta = json.load(f)
timestamps = np.load(trial0 / "screen" / "timestamps.npy")

stimuli = []
for key in sorted(combined_meta.keys(), key=lambda x: int(x)):
    e = combined_meta[key]
    if e["modality"] == "blank":
        continue
    onset_s = timestamps[e["first_frame_idx"]]
    if e["modality"] == "image":
        # discretize to the sim frame duration (matches RandomizedExperanto)
        pres_ms = FRAME_DURATION_MS * ((e["presentation_time"] * 1000) // FRAME_DURATION_MS)
        dur = pres_ms / 1000.0
        sid = f"img_{int(e['image_id'])}"
    else:  # video
        dur = e["num_frames"] * MOVIE_FRAME_DURATION_MS / 1000.0
        sid = f"vid_{e['condition_hash'][:8]}"
    stimuli.append(dict(key=key, stim_id=sid, modality=e["modality"],
                        onset_s=float(onset_s), offset_s=float(onset_s + dur), duration_s=dur))

n_img = sum(s["modality"] == "image" for s in stimuli)
n_vid = sum(s["modality"] == "video" for s in stimuli)
print(f"Found {len(stimuli)} stimuli: {n_img} images, {n_vid} videos")
for s in stimuli[:8]:
    print(f"  {s['stim_id']:22s} onset={s['onset_s']:.3f}s offset={s['offset_s']:.3f}s dur={s['duration_s']:.3f}s")

## 2. Load spikes for all trials (CSR: `spikes[indices[i]:indices[i+1]]` = neuron i)

In [ ]:
trial_data = []  # list of (spikes, indices)
for t in range(N_TRIALS):
    d = EXPERANTO_ROOT / f"trial{t}" / "responses"
    spikes = np.load(d / "spikes.npy")
    with open(d / "meta.yml") as f:
        indices = np.array(yaml.safe_load(f)["spike_indices"], dtype=np.int64)
    trial_data.append((spikes, indices))
    print(f"Trial {t}: {len(spikes):>10,} spikes, {len(indices) - 1} neurons, "
          f"range [{spikes.min():.3f}, {spikes.max():.3f}]s")

## 3. Pick the top-N most active neurons (by spike count in trial 0)

In [ ]:
_, indices0 = trial_data[0]
counts = np.diff(indices0)
top_neuron_indices = np.sort(np.argsort(counts)[::-1][:N_NEURONS])
total_s = max(sp.max() for sp, _ in trial_data)
print(f"Top {N_NEURONS} neurons (mean rate over {total_s:.1f}s):")
for idx in top_neuron_indices:
    print(f"  neuron {idx:5d}: {counts[idx]:>8,} spikes ({counts[idx] / total_s:5.1f} Hz)")

# Save for the datastore-PSTH step (§7) so it scores the SAME neurons in the SAME order.
np.save(OUTPUT_DIR / "top_neuron_indices.npy", top_neuron_indices)
print(f"\nSaved neuron indices -> {OUTPUT_DIR / 'top_neuron_indices.npy'}")

## 4. Per-neuron PSTH across trials

In [ ]:
def compute_stimulus_psth(trial_data, neuron_indices, onset_s, offset_s,
                          bin_size_ms=10, pre_ms=100, post_ms=100):
    """Return (psth[n_neurons, n_bins] Hz, sem[...], bin_edges_ms) averaged across trials."""
    bin_size_s = bin_size_ms / 1000.0
    ws, we = onset_s - pre_ms / 1000.0, offset_s + post_ms / 1000.0
    n_bins = int(np.ceil((we - ws) * 1000 / bin_size_ms))
    bin_edges_ms = np.arange(n_bins + 1) * bin_size_ms - pre_ms
    all_counts = np.zeros((len(neuron_indices), len(trial_data), n_bins))
    for ti, (spikes, indices) in enumerate(trial_data):
        for ni, nidx in enumerate(neuron_indices):
            ns = spikes[indices[nidx]:indices[nidx + 1]]
            lo = np.searchsorted(ns, ws, "left")
            hi = np.searchsorted(ns, we, "right")
            w = ns[lo:hi]
            if len(w):
                all_counts[ni, ti, :], _ = np.histogram((w - onset_s) * 1000.0, bins=bin_edges_ms)
    psth = all_counts.mean(axis=1) / bin_size_s
    sem = (all_counts.std(axis=1, ddof=1) / np.sqrt(len(trial_data))) / bin_size_s
    return psth, sem, bin_edges_ms


experanto_psths = {
    s["stim_id"]: compute_stimulus_psth(
        trial_data, top_neuron_indices, s["onset_s"], s["offset_s"],
        BIN_SIZE_MS, PRE_STIM_MS, POST_STIM_MS)
    for s in stimuli
}
print(f"Computed PSTHs for {len(experanto_psths)} stimuli.")

## 5. Verification — stimulus-locking (evoked vs. pre-stim baseline)

PASS if every stimulus shows population evoked rate > pre-stim baseline (spikes track the screen).

In [ ]:
print(f"{'stimulus':22s} {'baseline':>9s} {'evoked':>8s} {'ratio':>7s}   verdict")
locked = 0
for s in stimuli:
    psth, _, bins = experanto_psths[s["stim_id"]]
    pop = psth.mean(axis=0)
    centers = bins[:-1] + BIN_SIZE_MS / 2
    base = pop[centers < 0].mean()
    evoked = pop[(centers >= 0) & (centers <= s["duration_s"] * 1000)].mean()
    ratio = evoked / base if base > 0 else float("inf")
    ok = evoked > base
    locked += ok
    print(f"{s['stim_id']:22s} {base:9.2f} {evoked:8.2f} {ratio:7.2f}   {'LOCKED' if ok else 'flat'}")

print(f"\n{locked}/{len(stimuli)} stimuli stimulus-locked  ->  "
      f"{'VERIFY: PASS' if locked == len(stimuli) else f'PARTIAL ({locked}/{len(stimuli)})'}")

## 6. Plot population PSTHs (representative stimuli)

In [ ]:
rep = [s for s in stimuli if s["modality"] == "image"][:3] + \
      [s for s in stimuli if s["modality"] == "video"][:1]
fig, axes = plt.subplots(len(rep), 1, figsize=(11, 2.6 * len(rep)), squeeze=False)
for i, s in enumerate(rep):
    ax = axes[i, 0]
    psth, sem, bins = experanto_psths[s["stim_id"]]
    c = bins[:-1] + BIN_SIZE_MS / 2
    pop = psth.mean(axis=0)
    pop_sem = sem.mean(axis=0)
    ax.plot(c, pop, lw=1.5)
    ax.fill_between(c, pop - pop_sem, pop + pop_sem, alpha=0.2)
    ax.axvspan(0, s["duration_s"] * 1000, color="orange", alpha=0.15, label="stimulus")
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.set_title(f"{s['stim_id']} ({s['modality']})")
    ax.set_ylabel("Hz")
    ax.legend(fontsize=7)
axes[-1, 0].set_xlabel("time from onset (ms)")
fig.suptitle(f"Experanto export PSTHs — population mean (N={N_TRIALS} trials, top-{N_NEURONS} neurons)")
fig.tight_layout()
plt.show()

## 7. Integrity check — export PSTH vs. DataStore PSTH (ground truth)

The cells above verify the **export alone**. To confirm the export faithfully represents the source
MOZAIK **datastore**, compare the export PSTHs against PSTHs computed directly from the datastore
segments (same neurons, same binning).

Computing datastore PSTHs needs mozaik/neo, so it runs **in the container** as a separate step
(this notebook stays plain-numpy). Produce `datastore_psths.npz` once, then run the cell below.

```bash
# from the mozaik-models/experanto dir, inside the sim SIF (bind a mozaik checkout that has
# mozaik/tools/experanto_export.py, i.e. csng-mozaik-update / seed_refactor):
apptainer exec --cleanenv --env PYTHONPATH=/mozaik --env OMP_NUM_THREADS=1 \
  --env DATASTORE_DIR=<dir with SelfSustainedPushPull_trial*_chunk0_____*> \
  --env N_TRIALS=3 \
  --env EXPERANTO_META=<EXPERANTO_ROOT>/trial0/screen/combined_meta.json \
  --env NEURON_INDICES_FILE=<OUTPUT_DIR>/top_neuron_indices.npy \
  --env OUTPUT_DIR=<OUTPUT_DIR> \
  --bind <mozaik>:/mozaik --bind <experanto>:/project --bind <paths>:<paths> \
  <sim.sif> bash -lc '. /opt/mozaik/bin/activate; cd /project; \
                      python -u analysis/compute_psth_datastore.py'
```

`compute_psth_datastore.py` resolves datastore dirs by glob (any seed scheme), scores
`NEURON_INDICES_FILE` in `get_spiketrains()` order (= the export's neuron order), and writes
`datastore_psths.npz` to `OUTPUT_DIR`.

> Note: exact `exported spikes == datastore spikes` equality is separately guaranteed by
> `mozaik tests/tools/test_experanto_export.py`. This PSTH comparison is the response-level,
> timeline-aware cross-check (export screen-onsets vs. datastore segment-onsets).

In [ ]:
assert DATASTORE_PSTHS.exists(), (
    f"{DATASTORE_PSTHS} not found — run compute_psth_datastore.py first (see the cell above)."
)
ds = np.load(DATASTORE_PSTHS, allow_pickle=True)
print("neuron indices match export:", np.array_equal(ds["neuron_indices"], top_neuron_indices))

R_THRESHOLD = 0.95   # population-PSTH correlation required per stimulus
print(f"\n{'stimulus':22s} {'bins':>6s} {'Pearson r':>10s} {'max|Δ| Hz':>10s}   verdict")
rows, ok_all = [], True
for s in stimuli:
    sid = s["stim_id"]
    if f"{sid}_psth" not in ds:
        print(f"{sid:22s}  (missing in datastore npz)"); ok_all = False; continue
    ep = experanto_psths[sid][0]         # (neurons, bins) export
    dp = ds[f"{sid}_psth"]               # (neurons, bins) datastore
    n = min(ep.shape[1], dp.shape[1])    # tolerate ±1 bin from onset rounding
    e, d = ep[:, :n].ravel(), dp[:, :n].ravel()
    r = np.corrcoef(e, d)[0, 1]
    mad = float(np.max(np.abs(e - d)))
    ok = r >= R_THRESHOLD
    ok_all &= ok
    rows.append((s, ep, dp, n, r, mad))
    print(f"{sid:22s} {n:6d} {r:10.4f} {mad:10.2f}   {'OK' if ok else 'LOW'}")

print(f"\nINTEGRITY (export vs datastore PSTH): "
      f"{'PASS' if ok_all else 'CHECK'}  (all r >= {R_THRESHOLD}); "
      f"max|Δ| ~ a few spikes/bin is expected onset-rounding, not a fidelity issue.")

# overlay: population PSTH export vs datastore
rep = [r for r in rows if r[0]['modality'] == 'image'][:2] + \
      [r for r in rows if r[0]['modality'] == 'video'][:1]
fig, axes = plt.subplots(len(rep), 1, figsize=(11, 2.6 * len(rep)), squeeze=False)
for i, (s, ep, dp, n, r, mad) in enumerate(rep):
    ax = axes[i, 0]
    bins = experanto_psths[s["stim_id"]][2]
    c = (bins[:-1] + BIN_SIZE_MS / 2)[:n]
    ax.plot(c, ep[:, :n].mean(axis=0), lw=1.6, label="export")
    ax.plot(c, dp[:, :n].mean(axis=0), lw=1.0, ls="--", label="datastore")
    ax.axvspan(0, s["duration_s"] * 1000, color="orange", alpha=0.12)
    ax.axvline(0, color="k", ls=":", lw=0.8)
    ax.set_title(f"{s['stim_id']} ({s['modality']})  r={r:.4f}  max|Δ|={mad:.1f} Hz")
    ax.set_ylabel("Hz"); ax.legend(fontsize=8)
axes[-1, 0].set_xlabel("time from onset (ms)")
fig.suptitle(f"export vs datastore population PSTH (N={N_TRIALS} trials, top-{N_NEURONS} neurons)")
fig.tight_layout(); plt.show()